# 🛡️ Notebook 4 — Reliability Patterns

> **Goal:** In the first three notebooks we made an event bus that's **decoupled**, **async**, and has a **sane schema**.
> Production then asks three harder questions:
>
> 1. **What if the database write succeeds but the event publish fails?** → *Outbox pattern*
> 2. **What if step 3 of a 5-step workflow fails?** → *Saga with compensation*
> 3. **What if a single bad message keeps crashing our consumer?** → *Dead-letter queue*
>
> Each pattern is a tiny, runnable demo — no Kafka, no Rabbit, still < 40 lines each.


## 🛠️ Setup

```bash
cd 05-microservices/event-driven-architecture
uv sync
```

Then in VS Code, select the `.venv` kernel (top-right of the notebook).
If it isn't listed, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ The dual-write problem → the **Outbox pattern**

Imagine a classic "create order" endpoint:

```python
save_order_to_db(order)           # step A
bus.publish("OrderPlaced", order) # step B
```

What if **step A succeeds but step B crashes** (network blip, broker down, process killed)?
You now have an order in the DB that the rest of the world never hears about. Shipping, email, analytics — all silent. This is called the **dual-write problem**: two systems, no shared transaction.

### ❌ Naive version — let's break it on purpose


In [1]:
import random

# Tiny fake DB and fake broker
db = {}
published = []

def publish_flaky(event):
    # Pretend the network drops 60% of publishes.
    if random.random() < 0.6:
        raise RuntimeError("💥 broker unreachable")
    published.append(event)

def place_order_naive(order_id):
    db[order_id] = {"status": "placed"}           # ✅ step A
    publish_flaky({"type": "OrderPlaced",
                   "order_id": order_id})         # ❌ step B may fail

random.seed(7)
for i in range(5):
    try:
        place_order_naive(i)
        print(f"  order {i} — OK")
    except Exception as e:
        print(f"  order {i} — saved to DB but publish failed: {e}")

print(f"\nDB has {len(db)} orders, broker received {len(published)}.")
print("👉 Consumers think these orders never happened.")


  order 0 — saved to DB but publish failed: 💥 broker unreachable
  order 1 — saved to DB but publish failed: 💥 broker unreachable
  order 2 — OK
  order 3 — saved to DB but publish failed: 💥 broker unreachable
  order 4 — saved to DB but publish failed: 💥 broker unreachable

DB has 5 orders, broker received 1.
👉 Consumers think these orders never happened.


### ✅ The fix: write the event to an **outbox table in the same transaction**

Instead of publishing directly, we:

1. In **one DB transaction**, write the order *and* append the event to an `outbox` table.
2. A separate **relay** (a worker, or Debezium/CDC in real life) reads unpublished outbox rows and pushes them to the broker.
3. If publish fails, the row stays marked "unpublished" — the relay just tries again next tick.

The producer never has to coordinate with the broker. The DB transaction is the only atomic unit.


In [2]:
# Reset state
db, outbox, published = {}, [], []

def place_order_outbox(order_id):
    # Pretend this whole block is one DB transaction.
    db[order_id] = {"status": "placed"}
    outbox.append({
        "id": len(outbox),
        "type": "OrderPlaced",
        "order_id": order_id,
        "published": False,
    })
    # Note: NO publish call here. The relay handles it.

def relay_tick():
    # Run periodically. Pushes unpublished outbox rows to the broker.
    for row in outbox:
        if row["published"]:
            continue
        try:
            publish_flaky({"type": row["type"], "order_id": row["order_id"]})
            row["published"] = True           # only mark done on success
        except Exception as e:
            print(f"  relay: outbox #{row['id']} failed ({e}) — will retry")

# Producer runs; publishes never happen from its thread.
random.seed(7)
for i in range(5):
    place_order_outbox(i)

# Run the relay a few times — retries recover the failed publishes.
for tick in range(1, 10):
    print(f"— relay tick {tick}:")
    relay_tick()
    unpublished = sum(1 for r in outbox if not r["published"])
    print(f"  outbox unpublished={unpublished}, broker received={len(published)}")
    if unpublished == 0:
        print("  ✅ all caught up")
        break


— relay tick 1:
  relay: outbox #0 failed (💥 broker unreachable) — will retry
  relay: outbox #1 failed (💥 broker unreachable) — will retry
  relay: outbox #3 failed (💥 broker unreachable) — will retry
  relay: outbox #4 failed (💥 broker unreachable) — will retry
  outbox unpublished=4, broker received=1
— relay tick 2:
  relay: outbox #0 failed (💥 broker unreachable) — will retry
  relay: outbox #1 failed (💥 broker unreachable) — will retry
  relay: outbox #3 failed (💥 broker unreachable) — will retry
  relay: outbox #4 failed (💥 broker unreachable) — will retry
  outbox unpublished=4, broker received=1
— relay tick 3:
  relay: outbox #0 failed (💥 broker unreachable) — will retry
  relay: outbox #1 failed (💥 broker unreachable) — will retry
  relay: outbox #3 failed (💥 broker unreachable) — will retry
  relay: outbox #4 failed (💥 broker unreachable) — will retry
  outbox unpublished=4, broker received=1
— relay tick 4:
  relay: outbox #1 failed (💥 broker unreachable) — will retry
  re

### 🧠 What just happened

- **Zero events were lost**, even though the "network" was flaky.
- The producer's success condition is simpler: *"did the DB transaction commit?"*
- The relay's failure doesn't corrupt data — at worst it redelivers (hence your consumers must be idempotent, see notebook 3).

> In real systems you rarely hand-roll the relay. Tools like **Debezium**, **Kafka Connect**, or the **transactional outbox in Temporal** read your DB's change-log (`WAL` in Postgres, `binlog` in MySQL) and publish for you.


## 2️⃣ Multi-step workflows → the **Saga pattern**

A checkout is really *five* things, each in a different service:

1. Reserve inventory
2. Charge the card
3. Allocate a shipping label
4. Send the confirmation email
5. Give loyalty points

There's no distributed ACID transaction across five services. So what happens when **step 3 fails after steps 1 and 2 already committed**?

Answer: **compensating actions** run in reverse. "Release the inventory. Refund the card." Together the forward steps + their compensations form a **Saga**.

We'll implement it using the **mediator** style from notebook 3 because it makes the happy-path + rollback explicit.


In [3]:
class SagaStep:
    def __init__(self, name, do, undo):
        self.name, self.do, self.undo = name, do, undo

def run_saga(steps, ctx):
    completed = []
    for step in steps:
        try:
            print(f"  ▶️  {step.name}")
            step.do(ctx)
            completed.append(step)
        except Exception as e:
            print(f"  ❌ {step.name} failed: {e}")
            print(f"  ↩️  compensating in reverse order...")
            for done in reversed(completed):
                try:
                    done.undo(ctx)
                    print(f"     ✅ undid {done.name}")
                except Exception as ce:
                    # Compensations must be idempotent AND safe to retry.
                    print(f"     ⚠️  undo {done.name} failed: {ce} (log + alert)")
            return False
    return True


In [4]:
# --- forward actions ---
def reserve_inventory(ctx):  ctx['inventory'] = 'reserved'
def charge_card(ctx):        ctx['payment']   = 'captured'
def allocate_shipping(ctx):
    if ctx.get('force_fail'):
        raise RuntimeError('no carriers available')
    ctx['shipping']  = 'allocated'
def send_email(ctx):         ctx['email']     = 'sent'

# --- compensations (the "undo" for each forward step) ---
def release_inventory(ctx):  ctx['inventory'] = 'released'
def refund_card(ctx):        ctx['payment']   = 'refunded'
def cancel_shipping(ctx):    ctx['shipping']  = 'cancelled'
def _noop(ctx):              pass  # email has no meaningful undo

saga = [
    SagaStep('reserve_inventory', reserve_inventory, release_inventory),
    SagaStep('charge_card',       charge_card,       refund_card),
    SagaStep('allocate_shipping', allocate_shipping, cancel_shipping),
    SagaStep('send_email',        send_email,        _noop),
]

print('— Happy path:')
ctx = {}
ok = run_saga(saga, ctx)
print(f'  success={ok} state={ctx}\n')

print('— Shipping fails mid-way:')
ctx = {'force_fail': True}
ok = run_saga(saga, ctx)
print(f'  success={ok} state={ctx}')


— Happy path:
  ▶️  reserve_inventory
  ▶️  charge_card
  ▶️  allocate_shipping
  ▶️  send_email
  success=True state={'inventory': 'reserved', 'payment': 'captured', 'shipping': 'allocated', 'email': 'sent'}

— Shipping fails mid-way:
  ▶️  reserve_inventory
  ▶️  charge_card
  ▶️  allocate_shipping
  ❌ allocate_shipping failed: no carriers available
  ↩️  compensating in reverse order...
     ✅ undid charge_card
     ✅ undid reserve_inventory
  success=False state={'force_fail': True, 'inventory': 'released', 'payment': 'refunded'}


### 🧠 Design notes

- **Every forward step needs a compensating action** (except terminal/side-effect-free ones like sending email — you can't un-send an email, so put it last or accept "sorry, please ignore" emails).
- **Compensations are not the same as rollbacks.** A refund is a *new* business fact, not a deletion of the charge. Your books (and audit trail) still show both.
- **Compensations must be idempotent** — they may re-run if the saga orchestrator itself crashes.
- Real-world orchestrators: **Temporal**, **AWS Step Functions**, **Camunda**, **Netflix Conductor**. They add durability (the saga survives process crashes) and timers (step timeouts, retries).

There's also a **choreography** flavour where there's no orchestrator — each service listens for the previous step's success event and publishes its own. Simpler for small flows, harder to reason about as steps grow; see notebook 3 for the Broker vs Mediator tradeoff.


## 3️⃣ Poison messages → the **Dead-Letter Queue (DLQ)**

A single malformed event can take down a whole consumer if you retry forever:

- Attempt 1: crash, retry
- Attempt 2: crash, retry
- ...forever. The rest of the queue never moves. This is called **head-of-line blocking**.

The fix: after *N* attempts, **move the message to a side channel** (the DLQ) so humans can inspect it later, and keep draining the main queue.


In [5]:
from collections import deque

main_q = deque([
    {'id': 1, 'payload': {'amount': 10}},
    {'id': 2, 'payload': {'amount': 'oops'}},   # poison — not a number
    {'id': 3, 'payload': {'amount': 30}},
])
dlq = []
MAX_ATTEMPTS = 3

def process(msg):
    amount = msg['payload']['amount']
    if not isinstance(amount, (int, float)):
        raise TypeError(f"amount must be numeric, got {type(amount).__name__}")
    print(f"  ✅ processed id={msg['id']} total={amount * 2}")

attempts = {}
while main_q:
    msg = main_q.popleft()
    try:
        process(msg)
    except Exception as e:
        attempts[msg['id']] = attempts.get(msg['id'], 0) + 1
        if attempts[msg['id']] >= MAX_ATTEMPTS:
            print(f"  ☠️  id={msg['id']} moved to DLQ after {MAX_ATTEMPTS} attempts ({e})")
            dlq.append({'msg': msg, 'error': str(e)})
        else:
            print(f"  🔁 id={msg['id']} attempt {attempts[msg['id']]} failed, requeuing")
            main_q.append(msg)   # retry later

print(f"\nDLQ has {len(dlq)} poisoned message(s):")
for entry in dlq:
    print(' ', entry)


  ✅ processed id=1 total=20
  🔁 id=2 attempt 1 failed, requeuing
  ✅ processed id=3 total=60
  🔁 id=2 attempt 2 failed, requeuing
  ☠️  id=2 moved to DLQ after 3 attempts (amount must be numeric, got str)

DLQ has 1 poisoned message(s):
  {'msg': {'id': 2, 'payload': {'amount': 'oops'}}, 'error': 'amount must be numeric, got str'}


### 🧠 Why DLQs matter

- They **unblock the main queue** — good messages keep flowing.
- They **preserve evidence** for debugging — the original payload + error + attempt count.
- They turn a runtime outage into a **backlog an on-call can triage** later (fix the bug, replay from DLQ).

> Real brokers expose this out-of-the-box: **SQS redrive policy**, **Kafka DLT** (via Kafka Streams / Spring Cloud Stream), **RabbitMQ DLX**.


## 4️⃣ Beyond e-commerce — where else does this show up?

Same patterns, different industries. The event is always *"a business fact"*.

| Domain          | Producer publishes fact…     | Consumers care because…                                      |
|-----------------|------------------------------|--------------------------------------------------------------|
| **SaaS signup** | `UserSignedUp`               | welcome email · provision workspace · CRM lead · analytics   |
| **Ride-sharing**| `RideRequested`              | match driver · ETA · pricing · fraud check                   |
| **IoT**         | `SensorReadingReceived`      | store in timeseries DB · threshold alert · live dashboard    |
| **Banking**     | `TransactionPosted`          | update balance · fraud ML · receipt SMS · regulatory report  |
| **Gaming**      | `PlayerLeveledUp`            | cosmetics unlock · friend notifications · analytics          |
| **Healthcare**  | `LabResultReady`             | patient notify · physician inbox · billing · HL7 mirror      |

Notice the repetition of the recipe:

> **one fact → many independent reactions → add new reactions without touching the producer.**

That's the core promise of event-driven architecture.


## 🧪 Try it yourself

1. **Outbox:** add a `retry_count` to outbox rows and give up after N tries (move to an "outbox DLQ").
2. **Saga:** simulate `charge_card` failing *after* inventory is reserved. Check the compensation order in the output.
3. **DLQ:** write a `replay_dlq()` helper that moves DLQ entries back to `main_q` once they've been fixed.
4. Replace `publish_flaky` with a `redis.Redis().publish(...)` call. The outbox+relay structure doesn't change at all.

## ✅ Wrap up

You now have a mental model for:
- Bus mechanics (nb 1 & 2)
- Event design & topology (nb 3)
- **Reliability: outbox, saga, DLQ (this notebook)**

That's roughly the core curriculum for running event-driven systems in production without 3am pages.
